# ResNet Impairment-Aware Fine-Tuning — Freeze-Ratio Sweep (Teacher + Student)

Adapts Meneses-Albalá et al.'s U-Net fine-tuning idea (freeze a fraction of the network,
fine-tune the rest, on phase-impaired data) to this project's **ResNet** (Teacher, 64 blocks,
and the magnitude-pruned Student, also 64 blocks at width r=8). ResNet has no encoder/decoder
split (see `PROJECT_STATUS.md` discussion) — so instead of "freeze encoder, tune decoder,"
this freezes the first *K* of 64 (structurally identical) residual blocks and fine-tunes the
last (64−K), the standard deep-CNN transfer-learning analogue.

**Sweep:** freeze ratio K/64 ∈ {0%, 20%, 50%, 80%, 90%} (the paper's own 5 points) × 2 base
models (Teacher, Student) = **10 fine-tuning runs**, same code path for both.

**Run order:** first cell sets `SMOKE_TEST`. Leave it `True` and *Run All* first (a few
minutes) to confirm everything works, then set `False` and *Run All* for the real sweep.
Every run caches its result and checkpoints, so an interrupted run resumes on re-run.

## Kaggle setup

**Attach these datasets** (Notebook → Add Input):
- `dldoa-source-code` — `DL_DOA/` folder (evaluator + Teacher architecture code) and
  `dldoa_dataset_generation.py`. Same dataset used by every other notebook in this project.
- `dldoa-frozen-banks` — `eval_bank.npz`. Same dataset used by every other notebook.
- Whatever dataset has `student_r8_magnitude_lambda0.5.weights.h5` — the one already used in
  `DLDOA_Compression_04_NestedPathRobustness.ipynb`.

Everything else (Teacher weights `inf_model_007_256_resnet.h5`) is inside `dldoa-source-code`.
No GPU-selection step needed beyond Kaggle's own Settings → Accelerator → GPU; the code detects
it automatically. **Save Version → Save & Run All (Commit) when done**, or the outputs
(checkpoints, `RESULTS.md`) are lost when the session ends — this project has lost weights to
this exact mistake twice before.

In [ ]:
# ---------------------------------------------------------------- 1. Setup
import os, sys, json, time, math, random, traceback, datetime, zlib
os.environ.setdefault('TF_CPP_MIN_LOG_LEVEL', '2')

import numpy as np
import matplotlib
if 'ipykernel' not in sys.modules:
    matplotlib.use('Agg')
import matplotlib.pyplot as plt

try:
    import cv2
except ImportError:
    import subprocess
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'opencv-python-headless'], check=True)

import tensorflow as tf
from tensorflow.keras.layers import Conv2D, Input, BatchNormalization, Activation, Add, Conv2DTranspose
from tensorflow.keras.models import Model

tf.get_logger().setLevel('ERROR')
T_START = time.time()
GPUS = tf.config.list_physical_devices('GPU')
for g in GPUS:
    try:
        tf.config.experimental.set_memory_growth(g, True)
    except RuntimeError:
        pass
DEVICE = 'GPU' if GPUS else 'CPU'

# >>> Run mode: True = tiny sizes, checks the whole pipeline in minutes, writes to outputs_smoke/.
SMOKE_TEST = os.environ.get('FT_SMOKE', '0') == '1'


def find_root():
    """Local (this repo): walk up from cwd looking for frozen_banks/eval_bank.npz.
    Kaggle: no single root has everything (inputs are read-only, separate dataset mounts) --
    return /kaggle/working and let find_file() below search all of /kaggle/input too."""
    if os.path.isdir('/kaggle/input'):
        return '/kaggle/working'
    here = os.path.abspath(os.getcwd())
    for _ in range(4):
        if os.path.exists(os.path.join(here, 'frozen_banks', 'eval_bank.npz')):
            return here
        here = os.path.dirname(here)
    raise FileNotFoundError('Could not find the project root (a folder containing frozen_banks/eval_bank.npz).')


ON_KAGGLE = os.path.isdir('/kaggle/input')
ROOT = find_root()
OUT = os.path.join(ROOT, 'outputs_resnet_ft_smoke' if SMOKE_TEST else 'outputs_resnet_ft')
DIRS = {k: os.path.join(OUT, k) for k in ['results', 'figures', 'checkpoints', 'logs', 'tables']}
for d in DIRS.values():
    os.makedirs(d, exist_ok=True)
LOG_FILE = os.path.join(DIRS['logs'], 'run_log.txt')


def log(*a):
    msg = ' '.join(str(x) for x in a)
    line = f'[{(time.time() - T_START) / 60:7.1f} min] ' + msg
    print(line, flush=True)
    with open(LOG_FILE, 'a', encoding='utf-8') as f:
        f.write(line + '\n')


log('=' * 70)
log('ResNet impairment fine-tune suite start', datetime.datetime.now().isoformat(timespec='seconds'))
log(f'TF {tf.__version__} | device: {DEVICE} {GPUS} | ROOT={ROOT} | SMOKE_TEST={SMOKE_TEST}')
if not GPUS:
    log('WARNING: no GPU visible. TensorFlow >=2.11 has no native-Windows GPU support -- use Linux/WSL2.')

In [ ]:
# ---------------------------------------------------------------- 2. Config
CFG = dict(
    delta_max_deg=5.0,        # [ASSUMPTION] the one condition of the paper's {1,2,5} deg where it found
                              # a real, measurable gain (mild impairment: base model already robust)
    freeze_ratios=[0.0, 0.2, 0.5, 0.8, 0.9],   # paper's own 5 points, applied to "first K of 64 blocks"
    epochs=200,
    steps_per_epoch=40,       # [ASSUMPTION] paper gives batch size but not steps/epoch; fine-tuning from
                              # an already-converged model needs far fewer updates than from-scratch
                              # training, so this is picked for wall-clock, not copied from the paper
    batch=32,                 # paper Table 1
    lr_init=1e-4, lr_min=1e-6, lr_decay=0.75, plateau_patience=20,   # paper Table 1 (LR schedule)
    early_stop_patience=80,   # paper Table 1
    eval_every_epochs=20,     # cheap progress print (small subsample)
    eval_progress_n=60,       # samples for the per-20-epoch progress print
    eval_final_n=300,         # samples for the end-of-run Base-vs-Fine-tuned table (impaired + clean)
    ckpt_every=20,
)
if SMOKE_TEST:
    CFG.update(epochs=6, steps_per_epoch=3, batch=8, eval_every_epochs=2, eval_progress_n=6,
              eval_final_n=8, ckpt_every=2, plateau_patience=2, early_stop_patience=4)
log('CFG =', json.dumps(CFG))
with open(os.path.join(DIRS['logs'], 'config.json'), 'w') as f:
    json.dump(CFG, f, indent=2)

In [ ]:
# ---------------------------------------------------------------- 3. Result cache / resume
STATUS_PATH = os.path.join(DIRS['results'], 'status.json')
STATUS = json.load(open(STATUS_PATH)) if os.path.exists(STATUS_PATH) else {}


def _jsonable(o):
    if isinstance(o, dict):
        return {str(k): _jsonable(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)):
        return [_jsonable(v) for v in o]
    if isinstance(o, (np.floating, float)):
        return None if not math.isfinite(o) else float(o)
    if isinstance(o, (np.integer,)):
        return int(o)
    if isinstance(o, np.ndarray):
        return _jsonable(o.tolist())
    return o


def save_json(obj, path):
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(_jsonable(obj), f, indent=2)


def run_exp(name, fn):
    path = os.path.join(DIRS['results'], f'{name}.json')
    if os.path.exists(path):
        log(f'[{name}] cached')
        return json.load(open(path, encoding='utf-8'))
    log(f'[{name}] start')
    t0 = time.time()
    try:
        res = dict(fn() or {})
        res['_elapsed_min'] = (time.time() - t0) / 60
        save_json(res, path)
        STATUS[name] = f'done ({res["_elapsed_min"]:.1f} min)'
        log(f'[{name}] done in {res["_elapsed_min"]:.1f} min')
        save_json(STATUS, STATUS_PATH)
        return json.load(open(path, encoding='utf-8'))
    except Exception as e:
        with open(os.path.join(DIRS['logs'], 'errors.log'), 'a', encoding='utf-8') as f:
            f.write(f'\n===== {name} =====\n{traceback.format_exc()}\n')
        STATUS[name] = f'FAILED: {type(e).__name__}: {e}'
        save_json(STATUS, STATUS_PATH)
        log(f'[{name}] FAILED: {type(e).__name__}: {e} (see outputs*/logs/errors.log)')
        return None

In [ ]:
# ---------------------------------------------------------------- 4. Original project code
from pathlib import Path


def find_file(pattern):
    roots = ['/kaggle/input', '/kaggle/working', ROOT] if ON_KAGGLE else [ROOT]
    for root in roots:
        if not os.path.isdir(root):
            continue
        for p in Path(root).rglob(pattern):
            if p.is_file() and 'outputs' not in p.parts and 'IABR_Net_TestSuite' not in p.parts:
                return str(p)
    raise FileNotFoundError(f'{pattern} not found under {roots}' + (' -- attach the right Kaggle dataset(s), see the notebook\'s Cell 0' if ON_KAGGLE else ''))


DL_DOA_DIR = os.path.dirname(os.path.dirname(find_file('tvt_models.py')))
sys.path.insert(0, DL_DOA_DIR)
from src.tvt_models import Resnet
from src.TVT_Blob_Inference import (get_blob_detector, get_blob_peaks, peaks_to_angles,
                                    prepare_for_metric, get_ang_difference, filter_angles)
sys.path.insert(0, os.path.dirname(find_file('dldoa_dataset_generation.py')))
import dldoa_dataset_generation as DG

TEACHER_W = find_file('inf_model_007_256_resnet.h5')
STUDENT_W = find_file('student_r8_magnitude*')   # wildcard: matches regardless of the exact
                                                  # saved extension (.weights.h5 vs .h5), same
                                                  # pattern Notebook 4 already uses for this file
EVAL_BANK = find_file('eval_bank.npz')
log('evaluator imported from', DL_DOA_DIR)
log('teacher:', os.path.relpath(TEACHER_W, ROOT), '| student:', os.path.relpath(STUDENT_W, ROOT))

eval_bank = np.load(EVAL_BANK)
EVAL_DATA, EVAL_FEAT, EVAL_META = eval_bank['data'], eval_bank['feat'], eval_bank['meta']
SIGMA, M = float(eval_bank['sigma']), int(eval_bank['M'])

## Models and freeze mechanism

`get_weighted_layers` and the block layout (`stem, [conv1,bn1,conv2,bn2] x 64, final`) are the
exact pattern already used in `DLDOA_Compression_01/02` for this project's ResNet. Freezing sets
`.trainable = False` on the stem plus the first K blocks' 4 layers each; the standard TF training
loop (`tape.gradient(loss, model.trainable_variables)`) then only updates the unfrozen ones.

In [ ]:
# ---------------------------------------------------------------- 5. Model builders + freeze
def res_conv_pruned(x, r, out_filters=12):
    skip = x
    x = Conv2D(r, 5, padding='same')(x); x = BatchNormalization()(x); x = Activation('relu')(x)
    x = Conv2D(out_filters, 5, padding='same')(x); x = BatchNormalization()(x)
    x = Add()([x, skip]); x = Activation('relu')(x)
    return x


def build_pruned_resnet(r, n_blocks=64, name=None):
    x_in = Input(shape=(64, 64, 2))
    x = Conv2DTranspose(12, (5, 5), strides=(2, 2), padding='same')(x_in)
    for _ in range(n_blocks):
        x = res_conv_pruned(x, r)
    x = Conv2DTranspose(1, (5, 5), strides=(2, 2), padding='same')(x)
    return Model(x_in, x, name=name or f'ResNet-r{r}')


def get_weighted_layers(model):
    return [l for l in model.layers if l.get_weights()]


MODEL_SPECS = {
    'Teacher': dict(r=12, weights=TEACHER_W),
    'Student': dict(r=8, weights=STUDENT_W),
}


def load_base(name):
    spec = MODEL_SPECS[name]
    m = build_pruned_resnet(spec['r'], name=name) if name == 'Student' else Resnet(input_shape=(64, 64, 2))
    m.load_weights(spec['weights'])
    return m


def set_freeze(model, freeze_ratio, n_blocks=64):
    """Freeze the stem + first K blocks (K = round(freeze_ratio*64)); leave the rest (+ final
    layer) trainable. K=0 (the paper's 0% condition) freezes nothing at all -- a true full
    fine-tune, matching "Encoder frozen at 0%" in their Table 1. Returns K.

    VERIFIED locally (see debug_freeze.py) before this was trusted: an earlier version of this
    function had the boolean inverted (`i < n_frozen_layers` instead of `i >= `), which froze
    EVERYTHING except the stem regardless of K -- caught because a K=0 smoke run printed
    trainable_params=612 (exactly the stem's own param count) instead of ~full model."""
    layers = get_weighted_layers(model)
    assert len(layers) == 1 + 4 * n_blocks + 1, f'unexpected layer count {len(layers)}'
    K = round(freeze_ratio * n_blocks)
    n_frozen_layers = (1 + 4 * K) if K > 0 else 0   # stem + K blocks, or nothing at K=0
    for i, l in enumerate(layers):
        l.trainable = (i >= n_frozen_layers)
    n_trainable_params = int(sum(np.prod(v.shape) for v in model.trainable_variables))
    return K, n_trainable_params


for name in MODEL_SPECS:
    m = load_base(name)
    log(f'{name}: {m.count_params():,} total params, {len(get_weighted_layers(m))} weighted layers')
    del m; tf.keras.backend.clear_session()

## Impairment-augmented training data

Exactly the project's own `training_data_generator`, with one change: the TX/RX codebooks are
built with `error_deg=DELTA_MAX` instead of `error_deg=None`. `beamforming_vector_generation_P/Q`
already draws a **fresh** per-antenna phase error `~ U(-delta_max, delta_max)` on every call (see
`dldoa_dataset_generation.py`), so every training sample gets an independent impairment draw —
matching the paper's fine-tuning-data description. Fine-tuning trains on 100% impaired data (like
the paper); the frozen early layers are what protects against forgetting the clean distribution.

In [ ]:
# ---------------------------------------------------------------- 6. Data generators
def impaired_training_batch(rng, batch, delta_max, sigma=0.07, M=256):
    xs, ys = [], []
    for _ in range(batch):
        L = rng.integers(1, 10)
        SNR = rng.integers(-15, 25)
        P = int(rng.choice([16, 32])); Q = P
        if P == 32:
            nt = int(rng.choice([16, 32]))
        else:
            nt = 16
        nr = nt
        F = DG.beamforming_vector_generation_P(P, nt, error_deg=delta_max)
        W = DG.beamforming_vector_generation_Q(Q, nr, error_deg=delta_max)
        alpha_l = (np.sqrt(1 / L) * (rng.standard_normal(L) + 1j * rng.standard_normal(L)) / np.sqrt(2))
        alpha_l = alpha_l[np.argsort(-np.abs(alpha_l))]
        pts = DG.generate_points(L, np.pi / 6, rng=rng)
        phi_l = np.array([p[0] for p in pts]); psi_l = np.array([p[1] for p in pts])
        angle_v = np.hstack([phi_l, psi_l])
        omega_phi = np.pi * np.cos(phi_l); omega_psi = -np.pi * np.cos(psi_l)
        H = DG.generate_channel_v2(nr, nt, angle_v, alpha_l)
        G = (W.conj().T @ H) @ F   # == W.view(_myarray).H @ H, the original's Hermitian-transpose helper
        Z = DG.generate_noise(1.0, SNR, P, Q, rng=rng)
        Y = DG.get_real_imag(G + Z)
        gt = DG.generate_gt(L, np.ones(L), omega_phi, omega_psi, num_points_rx=M, num_points_tx=M, sigma=sigma)
        zf = 4 if P == 16 else 2
        import scipy.ndimage
        data = np.dstack([scipy.ndimage.zoom(Y[:, :, 0], zf, order=0), scipy.ndimage.zoom(Y[:, :, 1], zf, order=0)])
        xs.append(np.real(data).astype(np.float32)); ys.append(np.real(gt)[..., None].astype(np.float32))
    return np.stack(xs), np.stack(ys)


def make_train_ds(delta_max, batch, seed):
    def gen():
        rng = np.random.default_rng(seed)
        while True:
            yield impaired_training_batch(rng, batch, delta_max, sigma=SIGMA, M=M)
    ds = tf.data.Dataset.from_generator(
        gen, output_signature=(tf.TensorSpec([batch, 64, 64, 2], tf.float32), tf.TensorSpec([batch, M, M, 1], tf.float32)))
    return ds.prefetch(2)


def eval_bank_impaired(n_per_snr, delta_max, seed):
    """Fixed, reproducible impaired eval samples, same convention as validation_data_generator
    (L=3, the project's standard eval configuration) across all 8 SNR points."""
    conds = [(3, snr, 16, 16) for snr in range(-10, 30, 5)]
    gen = DG.validation_data_generator(conds, examples_per_condition=n_per_snr, seed=seed, sigma=SIGMA, M=M, error_deg=delta_max)
    data, feat, meta = [], [], []
    for d, _, f, m in gen:
        data.append(d); feat.append(f); meta.append(m)
    return np.stack(data), np.stack(feat), np.stack(meta)

## Evaluation (original evaluator, same convention as every other notebook)

In [ ]:
# ---------------------------------------------------------------- 7. Evaluator
DETECTOR = get_blob_detector()


def evaluate_on(model, data, feat, meta, batch_size=16):
    results_by_snr = {}
    for s in range(0, len(data), batch_size):
        pred = model(data[s:s + batch_size], training=False)
        for j in range(pred.shape[0]):
            i = s + j; L = int(meta[i, 0]); snr = int(meta[i, 1])
            pk, am = get_blob_peaks(pred[j], DETECTOR); pk = pk[np.argsort(-am)[:L]]
            angles = peaks_to_angles(pk, sigma=SIGMA, grid_size=M)
            gt, pr = prepare_for_metric(angles, feat[i])
            results_by_snr.setdefault(snr, []).append((gt, pr))
    pd_, rmse_ = {}, {}
    for snr, ex in results_by_snr.items():
        good, bad = [], []
        for gt, pr in ex:
            if np.isnan(pr).any():
                continue
            g, b = filter_angles(get_ang_difference(gt, pr), 1.0)
            good.append(g); bad.append(b)
        good = np.concatenate(good) if good else np.array([]); bad = np.concatenate(bad) if bad else np.array([])
        tot = len(good) + len(bad)
        pd_[snr] = len(good) / tot if tot else np.nan
        rmse_[snr] = float(np.sqrt(np.mean(good ** 2))) if len(good) else np.nan
    return pd_, rmse_


def mean_pd(d):
    return float(np.nanmean(list(d.values())))

## Fine-tuning loop

Per-epoch training loss is always logged (cheap). A small (`eval_progress_n`-sample) accuracy
check runs every `eval_every_epochs` and prints Pd so you can watch progress without paying for a
full evaluation every epoch. LR follows the paper's plateau schedule (on training loss, since a
true validation-loss signal would need a full blob-detection pass every epoch); early stopping
and "best checkpoint" use the same training-loss signal.

In [ ]:
# ---------------------------------------------------------------- 8. Fine-tune one (model, freeze_ratio)
def finetune_run(model_name, freeze_ratio, delta_max, epochs, steps_per_epoch, batch, run_key):
    cdir = os.path.join(DIRS['checkpoints'], run_key)
    os.makedirs(cdir, exist_ok=True)
    done_path = os.path.join(cdir, 'done.json'); best_path = os.path.join(cdir, 'best.weights.h5')
    final_path = os.path.join(cdir, 'final.weights.h5'); hist_path = os.path.join(cdir, 'history.json')
    if os.path.exists(done_path):
        return json.load(open(done_path))

    model = load_base(model_name)
    K, n_trainable = set_freeze(model, freeze_ratio)
    log(f'  [{run_key}] freeze {freeze_ratio*100:.0f}% -> K={K}/64 blocks frozen, {n_trainable:,} trainable params')

    opt = tf.keras.optimizers.Adam(CFG['lr_init'])
    step_var = tf.Variable(0, dtype=tf.int64)
    ckpt = tf.train.Checkpoint(model=model, optimizer=opt, step=step_var)
    mgr = tf.train.CheckpointManager(ckpt, os.path.join(cdir, 'ckpt'), max_to_keep=2)
    history = json.load(open(hist_path)) if os.path.exists(hist_path) else []
    if mgr.latest_checkpoint:
        ckpt.restore(mgr.latest_checkpoint)
        log(f'  [{run_key}] resumed at epoch {len(history)}')

    @tf.function
    def train_step(x, y):
        with tf.GradientTape() as tape:
            pred = model(x, training=True)
            loss = tf.reduce_mean(tf.square(pred - y))
        grads = tape.gradient(loss, model.trainable_variables)
        opt.apply_gradients(zip(grads, model.trainable_variables))
        return loss

    # deterministic across process restarts (unlike builtin hash(), which is randomized per process
    # unless PYTHONHASHSEED=0) -- keeps a resumed run's data stream reproducible
    seed = zlib.crc32(run_key.encode()) % (2 ** 31)
    ds_iter = iter(make_train_ds(delta_max, batch, seed=seed))
    start_epoch = len(history)
    if history:
        best_idx = min(range(len(history)), key=lambda i: history[i]['loss'])
        best_loss = history[best_idx]['loss']
        epochs_since_best = len(history) - 1 - best_idx
        lr = history[-1]['lr']
    else:
        best_loss = float('inf'); epochs_since_best = 0; lr = CFG['lr_init']
    plateau_ctr = 0
    t0 = time.time(); eta_logged = start_epoch > 0

    for epoch in range(start_epoch, epochs):
        opt.learning_rate.assign(lr)
        ep_t0 = time.time()
        losses = [float(train_step(*next(ds_iter))) for _ in range(steps_per_epoch)]
        ep_loss = float(np.mean(losses))
        history.append(dict(epoch=epoch + 1, loss=ep_loss, lr=lr, sec=time.time() - ep_t0))
        step_var.assign_add(steps_per_epoch)

        if not eta_logged:
            sps = time.time() - t0
            log(f'  [{run_key}] measured {sps:.2f}s/epoch -> ETA {sps * (epochs - epoch - 1) / 60:.1f} min for this run')
            eta_logged = True

        if ep_loss < best_loss - 1e-6:
            best_loss = ep_loss; plateau_ctr = 0; epochs_since_best = 0
            model.save_weights(best_path)
        else:
            plateau_ctr += 1; epochs_since_best += 1
            if plateau_ctr >= CFG['plateau_patience']:
                lr = max(CFG['lr_min'], lr * CFG['lr_decay']); plateau_ctr = 0
                log(f'  [{run_key}] epoch {epoch+1}: LR plateau -> {lr:.2e}')

        if (epoch + 1) % 5 == 0 or epoch == start_epoch:
            log(f'  [{run_key}] epoch {epoch+1}/{epochs}: loss={ep_loss:.5f} lr={lr:.2e} best={best_loss:.5f} ({time.time()-ep_t0:.1f}s)')

        if (epoch + 1) % CFG['eval_every_epochs'] == 0 or epoch + 1 == epochs:
            d, f, m = eval_bank_impaired(max(1, CFG['eval_progress_n'] // 8), delta_max, seed=999)
            pd_, _ = evaluate_on(model, d, f, m)
            log(f'  [{run_key}] epoch {epoch+1}: progress-eval mean Pd (impaired, {len(d)} samples) = {mean_pd(pd_):.4f}')

        if (epoch + 1) % CFG['ckpt_every'] == 0:
            mgr.save(); save_json(history, hist_path)

        if epochs_since_best >= CFG['early_stop_patience']:
            log(f'  [{run_key}] early stop at epoch {epoch+1} (no improvement for {CFG["early_stop_patience"]} epochs)')
            break

    model.save_weights(final_path)
    save_json(history, hist_path)
    if not os.path.exists(best_path):
        model.save_weights(best_path)
    info = dict(run_key=run_key, model=model_name, freeze_ratio=freeze_ratio, K_frozen=K,
               n_trainable_params=n_trainable, delta_max_deg=delta_max, epochs_run=len(history),
               best_loss=best_loss, train_min=(time.time() - t0) / 60)
    save_json(info, done_path)
    del model; tf.keras.backend.clear_session()
    return info

## Run the sweep: 2 models x 5 freeze ratios = 10 fine-tuning runs

After each run, the fine-tuned model (best checkpoint) is compared against the **unmodified base
model** on the same fixed impaired eval set, and also on a clean (no-impairment) eval set to check
for forgetting — printed immediately so progress is visible run by run.

In [ ]:
# ---------------------------------------------------------------- 9. Sweep
RUNS = [(model, fr) for model in MODEL_SPECS for fr in CFG['freeze_ratios']]
log(f'{len(RUNS)} fine-tuning runs: ' + ', '.join(f'{m}@{int(fr*100)}%' for m, fr in RUNS))

EVAL_IMP_D, EVAL_IMP_F, EVAL_IMP_M = eval_bank_impaired(CFG['eval_final_n'] // 8, CFG['delta_max_deg'], seed=42)
EVAL_CLEAN_D, EVAL_CLEAN_F, EVAL_CLEAN_M = eval_bank_impaired(CFG['eval_final_n'] // 8, None, seed=43)
log(f'Final-eval banks ready: {len(EVAL_IMP_D)} impaired samples (delta_max={CFG["delta_max_deg"]} deg), '
    f'{len(EVAL_CLEAN_D)} clean samples')

BASE_CACHE = {}


def base_scores(model_name):
    if model_name not in BASE_CACHE:
        m = load_base(model_name)
        imp_pd, imp_rmse = evaluate_on(m, EVAL_IMP_D, EVAL_IMP_F, EVAL_IMP_M)
        clean_pd, clean_rmse = evaluate_on(m, EVAL_CLEAN_D, EVAL_CLEAN_F, EVAL_CLEAN_M)
        BASE_CACHE[model_name] = dict(impaired_pd=imp_pd, impaired_rmse=imp_rmse, clean_pd=clean_pd, clean_rmse=clean_rmse)
        del m; tf.keras.backend.clear_session()
        log(f'  base {model_name}: impaired mean Pd={mean_pd(imp_pd):.4f} | clean mean Pd={mean_pd(clean_pd):.4f}')
    return BASE_CACHE[model_name]


ALL_ROWS = []
for model_name, fr in RUNS:
    run_key = f'{model_name}_freeze{int(fr*100):02d}'
    base = run_exp(f'BASE_{model_name}', lambda mn=model_name: {'scores': base_scores(mn)})
    info = run_exp(f'FT_{run_key}', lambda mn=model_name, r=fr, rk=run_key: finetune_run(
        mn, r, CFG['delta_max_deg'], CFG['epochs'], CFG['steps_per_epoch'], CFG['batch'], rk))
    if info is None:
        continue
    ft = load_base(model_name)
    ft.load_weights(os.path.join(DIRS['checkpoints'], run_key, 'best.weights.h5'))
    imp_pd, imp_rmse = evaluate_on(ft, EVAL_IMP_D, EVAL_IMP_F, EVAL_IMP_M)
    clean_pd, clean_rmse = evaluate_on(ft, EVAL_CLEAN_D, EVAL_CLEAN_F, EVAL_CLEAN_M)
    del ft; tf.keras.backend.clear_session()
    b = base_scores(model_name)
    row = dict(model=model_name, freeze_pct=int(fr * 100), K=info['K_frozen'], trainable_params=info['n_trainable_params'],
              epochs_run=info['epochs_run'],
              base_impaired_pd=mean_pd(b['impaired_pd']), ft_impaired_pd=mean_pd(imp_pd),
              d_impaired_pd=mean_pd(imp_pd) - mean_pd(b['impaired_pd']),
              base_clean_pd=mean_pd(b['clean_pd']), ft_clean_pd=mean_pd(clean_pd),
              d_clean_pd=mean_pd(clean_pd) - mean_pd(b['clean_pd']),
              base_impaired_rmse=float(np.nanmean(list(b['impaired_rmse'].values()))),
              ft_impaired_rmse=float(np.nanmean(list(imp_rmse.values()))))
    ALL_ROWS.append(row)
    log(f'[{run_key}] RESULT: impaired Pd {row["base_impaired_pd"]:.4f} -> {row["ft_impaired_pd"]:.4f} '
        f'(delta {row["d_impaired_pd"]:+.4f}) | clean Pd {row["base_clean_pd"]:.4f} -> {row["ft_clean_pd"]:.4f} '
        f'(delta {row["d_clean_pd"]:+.4f}) | trainable {row["trainable_params"]:,}/{{469393,314513}}[model]')
    save_json(ALL_ROWS, os.path.join(DIRS['results'], 'sweep_rows.json'))

In [ ]:
# ---------------------------------------------------------------- 10. Tables + figures
def md_table(rows, cols, fmt=None):
    fmt = fmt or {}
    head = '| ' + ' | '.join(cols) + ' |\n|' + '---|' * len(cols) + '\n'
    body = ''
    for r in rows:
        cells = []
        for c in cols:
            v = r.get(c, '')
            if isinstance(v, float):
                v = '—' if not math.isfinite(v) else fmt.get(c, '{:.4f}').format(v)
            elif isinstance(v, int) and c in fmt:
                v = fmt[c].format(v)
            cells.append(str(v))
        body += '| ' + ' | '.join(cells) + ' |\n'
    return head + body


COLS = ['model', 'freeze_pct', 'K', 'trainable_params', 'epochs_run', 'base_impaired_pd', 'ft_impaired_pd',
        'd_impaired_pd', 'base_clean_pd', 'ft_clean_pd', 'd_clean_pd']
TABLE_MD = md_table(ALL_ROWS, COLS, fmt={'trainable_params': '{:,}'})
with open(os.path.join(DIRS['tables'], 'sweep.md'), 'w', encoding='utf-8') as f:
    f.write(f'### Freeze-ratio sweep — Base vs Fine-tuned (delta_max={CFG["delta_max_deg"]} deg, '
            f'{len(EVAL_IMP_D)} impaired + {len(EVAL_CLEAN_D)} clean eval samples)\n\n{TABLE_MD}\n')
log('\n' + TABLE_MD)

if ALL_ROWS and not SMOKE_TEST:
    fig, axs = plt.subplots(1, 2, figsize=(13, 4.5))
    for model_name in MODEL_SPECS:
        rows = [r for r in ALL_ROWS if r['model'] == model_name]
        if not rows:
            continue
        axs[0].plot([r['freeze_pct'] for r in rows], [r['d_impaired_pd'] for r in rows], marker='o', label=model_name)
        axs[1].plot([r['freeze_pct'] for r in rows], [r['d_clean_pd'] for r in rows], marker='o', label=model_name)
    axs[0].set_title('Impaired-data gain (fine-tuned - base mean Pd)'); axs[0].axhline(0, color='gray', lw=.7)
    axs[1].set_title('Clean-data change (forgetting check)'); axs[1].axhline(0, color='gray', lw=.7)
    for a in axs:
        a.set_xlabel('freeze ratio (%)'); a.set_ylabel('delta mean Pd'); a.grid(alpha=.3); a.legend()
    plt.tight_layout(); plt.savefig(os.path.join(DIRS['figures'], 'freeze_sweep.png'), dpi=130); plt.show()

    plt.figure(figsize=(8, 4.5))
    for model_name, fr in RUNS:
        run_key = f'{model_name}_freeze{int(fr*100):02d}'
        hp = os.path.join(DIRS['checkpoints'], run_key, 'history.json')
        if os.path.exists(hp):
            h = json.load(open(hp))
            plt.plot([x['epoch'] for x in h], [x['loss'] for x in h], label=run_key, alpha=.8)
    plt.xlabel('epoch'); plt.ylabel('training loss'); plt.legend(fontsize=6, ncol=2); plt.grid(alpha=.3)
    plt.title('Per-epoch training loss, all runs'); plt.tight_layout()
    plt.savefig(os.path.join(DIRS['figures'], 'loss_curves.png'), dpi=130); plt.show()

In [ ]:
# ---------------------------------------------------------------- 11. Final report
lines = ['# ResNet impairment fine-tuning — results', '',
         f'- Generated: {datetime.datetime.now().isoformat(timespec="seconds")}',
         f'- Device: {DEVICE} {GPUS} | TF {tf.__version__} | SMOKE_TEST: **{SMOKE_TEST}**',
         f'- delta_max = {CFG["delta_max_deg"]} deg | epochs={CFG["epochs"]} steps/epoch={CFG["steps_per_epoch"]} batch={CFG["batch"]}',
         f'- Total wall-clock: {(time.time()-T_START)/3600:.2f} h', '',
         '## Status', '', '| run | status |', '|---|---|']
lines += [f'| {k} | {v} |' for k, v in STATUS.items()]
lines += ['', TABLE_MD if ALL_ROWS else '(no runs completed)', '',
         '## Figures', ''] + [f'- `figures/{f}`' for f in sorted(os.listdir(DIRS['figures']))]
with open(os.path.join(OUT, 'RESULTS.md'), 'w', encoding='utf-8') as f:
    f.write('\n'.join(lines))
log('RESULTS written to', os.path.relpath(os.path.join(OUT, 'RESULTS.md'), ROOT))
log(f'total session time {(time.time()-T_START)/3600:.2f} h')